In [3]:
import pandas as pd
import matplotlib as plt

# Load data into dfs

In [4]:
hockey_scouting_notes = pd.read_csv('../data/hockey_scouting_notes.csv')   # joinable by international_id
contracts_competition = pd.read_pickle('../data/contracts_competition.pkl') # joinable by international_id
performance = pd.read_csv('../data/performance.tsv', sep='\t') # joinable by international_id

identity_card_0 = pd.read_csv('../data/identity_card_0.tsv', sep='\t', header=0, names=['international_id',
                                                                                'medical_id',
                                                                                'first_name',
                                                                                'last_name',
                                                                                'gender',
                                                                                'age',
                                                                                'birth_city',
                                                                                'nationality'])
identity_card_1 = pd.read_csv('../data/identity_card_1.csv')

medical_information = pd.read_excel('../data/medical_information.xlsx',
                                    skiprows=2,
                                    header=0, 
                                    names=['medical_id',
                                           'height',
                                           'weight',
                                           'age_in_years',
                                           'shoe_size',
                                           'body_fat_percentage',
                                           'fitness_level',
                                           'sprint_time',
                                           'medical_information',
                                           'return_date',
                                           'physician_signature']).drop(columns=['shoe_size', 
                                                                                 'return_date', 
                                                                                 'physician_signature']) # joinable by medical_id

moms_notes = pd.read_json('../data/moms_notes.json')



# Data cleaning and processing

## Identity card

In [5]:
identity_card_1['gender'].unique()

<StringArray>
['male', 'other', 'female', 'f', 'm', nan]
Length: 6, dtype: str

In [6]:
def transfer_roman_to_int(roman_numeral):
    if pd.isna(roman_numeral) or not isinstance(roman_numeral, str):
        return roman_numeral
        
    roman_map = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    prev_value = 0
    for char in reversed(roman_numeral.upper().strip()):
        value = roman_map.get(char, 0)
        if value < prev_value:
            total -= value
        else:
            total += value
        prev_value = value
    return total

identity_card_0['international_id']=identity_card_0['international_id'].apply(transfer_roman_to_int)


In [7]:
# Take out the ius from last names an "us" from "male" first names
identity_card_0['last_name'] = identity_card_0['last_name'].apply(lambda x: x[:-3])
identity_card_0['first_name'] = identity_card_0.apply(lambda x: x['first_name'][:-2] if x['gender'] == 'male' else x['first_name'],axis=1)

In [8]:
identity_card = pd.concat([identity_card_0,identity_card_1])

# international_id

# gender -> normalize 
gender_map = {'f':'female',
              'female': 'female',
              
              'm': 'male',
              'male': 'male',
              
              'other': 'other'}

identity_card['gender'] = identity_card['gender'].map(gender_map)

# age (why is there someone 312 years old?)

# birth_city and nationality -> normalize 
identity_card['birth_city'] = identity_card['birth_city'].str.lower().str.strip()
identity_card['nationality'] = identity_card['nationality'].str.lower().str.strip()


identity_card


,international_id,medical_id,first_name,last_name,gender,age,birth_city,nationality
0,275,039e0ab7-e36d-4bc7-a0b1-3e93e8ec73e9,Keith,Allee,male,23.0,vienna,austria
1,2560,070dfb40-76c9-4894-90e5-7463d060d49b,Alyssa,Kohl,female,31.0,vienna,austria
2,2323,fa06ed07-6f09-4ec2-9f11-a70917307270,Emily,Belanger,female,29.0,ostrava,france
3,7708,59f94605-e953-4117-b9ee-09ba1392e895,Stefanie,Fiedler,female,25.0,rome,italy
4,4062,2e663575-2f96-42fc-ab59-481993a65a53,Melanie,Mulligan,female,312.0,paris,france
...,...,...,...,...,...,...,...,...
9107,2660,94d60877-0aee-4318-aa6d-3deae0cb78fa,Walker,Rice,male,25.0,new york,usa
9108,4550,26d12241-ad53-418b-9b07-a0bacbf24213,Barbara,Ramsey,female,23.0,daugavpils,norway
9109,6981,d2ad5fdd-6f48-4395-b901-bd107f0ed85c,David,Pawlicki,male,18.0,boston,usa
9110,1249,b8822ca3-be3c-49a5-91be-ab701f562f8e,Sharon,Hancock,female,26.0,munich,germany


### ID card cleanup actions and remarks

* International Ids from id_card_0 transfered from roman to arab numerals
* From id_card_0 take out "ius" in last name
* From id_card_0 take out "us" in first names of males (One guys name was Hilarious before and now its only Hilario, which is a bit weird but i don't even know with this dataset) :D
* Normalize gender mapping
* Normalize cities and countries to lowercase

Remarks
* Some age outliers, what to do with that??? diverse and synthesized dataset, leave old people in 
* Birth cities and nationalities often don't match
* There are 37 random rows with no data apart from international_id and medical_id, I'd drop those.

TODO
* Drop brith city because nationality and birth city doesnt match, not important
* drop 37 random rows
* drop people older than the oldest person on earth

## Scouting notes

In [9]:
# fill nan in scout_notes with no notes
hockey_scouting_notes['scout_notes'] = hockey_scouting_notes['scout_notes'].fillna('no notes')

#fill nan values in dominant_hand with none
hockey_scouting_notes['dominant_hand'] = hockey_scouting_notes['dominant_hand'].fillna('none')

#drop people with 0 years + not rookie
rows = hockey_scouting_notes[(hockey_scouting_notes['years_played'] == 0) & (hockey_scouting_notes['experience_level'] != 'rookie')].index
hockey_scouting_notes.drop(rows, inplace=True)

#drop people who have more year played pro than years played
hockey_scouting_notes = hockey_scouting_notes[hockey_scouting_notes['years_played'] >= hockey_scouting_notes['years_pro']]
hockey_scouting_notes


hockey_scouting_notes

,international_id,position,dominant_hand,experience_level,years_played,years_pro,scout_notes
0,2991,defense,left,veteran,8.0,8.0,"effort varies throughout the game, awkward str..."
1,3042,right wing,right,veteran,0.0,0.0,a consistent performer who can be counted on i...
2,3761,right wing,right,veteran,0.0,0.0,late support on breakouts
3,6229,left wing,left,rookie,1.0,1.0,NaN
4,1303,defense,NaN,veteran,9.0,9.0,"poor awareness in defensive zone, strong and a..."
...,...,...,...,...,...,...,...
9995,6627,left wing,right,sophomore,1.0,1.0,refuses to change stick after a goal
9996,5202,defense,left,veteran,10.0,5.0,sets the tone early in games
9997,2182,goalie,left,veteran,1.0,1.0,combines intelligence with strong leadership q...
9998,3387,defense,left,rookie,6.0,6.0,shoots from anywhere just in case


### Scouting notes cleanup actions and remarks

Remarks
* Again some missing values, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before

TODOS
* delete 37 random rows
* fill NaN in scout_notes with "no notes"
* fill NaN in dominant_hand with "none"
* drop people with 0 years + veteran 
* check if years_pro is more than years_played --> drop 

## Contracts competition

In [10]:
contracts_competition

,international_id,contracts_signed,salary,captain,won_championship,jersey_number,draft_year,number_of_previous_teams
0,6032,5.0,669578.77,False,True,48.0,2022.0,12.0
1,3654,5.0,1647102.80,False,False,10.0,2013.0,15.0
2,8729,6.0,1678593.48,False,False,63.0,2016.0,14.0
3,2300,1.0,927858.79,False,False,46.0,2015.0,8.0
4,8407,2.0,710328.25,True,True,79.0,2012.0,15.0
...,...,...,...,...,...,...,...,...
9995,4579,5.0,490417.83,False,False,7.0,2020.0,14.0
9996,8411,4.0,3389293.11,True,False,95.0,2015.0,15.0
9997,8570,12.0,1221299.65,False,False,1.0,2010.0,10.0
9998,2010,6.0,2279861.54,False,False,3.0,2017.0,8.0


In [11]:
# Cleaning contracts_competition

# contracts_signed, jersey_number, draft_year, number_of_previous_teams -> convert to int
contracts_competition['contracts_signed'] = contracts_competition['contracts_signed'].astype("Int64")
contracts_competition['jersey_number'] = contracts_competition['jersey_number'].astype("Int64")
contracts_competition['draft_year'] = contracts_competition['draft_year'].astype("Int64")
contracts_competition['number_of_previous_teams'] = contracts_competition['number_of_previous_teams'].astype("Int64")

# jersey_number
contracts_competition

,international_id,contracts_signed,salary,captain,won_championship,jersey_number,draft_year,number_of_previous_teams
0,6032,5,669578.77,False,True,48,2022,12
1,3654,5,1647102.80,False,False,10,2013,15
2,8729,6,1678593.48,False,False,63,2016,14
3,2300,1,927858.79,False,False,46,2015,8
4,8407,2,710328.25,True,True,79,2012,15
...,...,...,...,...,...,...,...,...
9995,4579,5,490417.83,False,False,7,2020,14
9996,8411,4,3389293.11,True,False,95,2015,15
9997,8570,12,1221299.65,False,False,1,2010,10
9998,2010,6,2279861.54,False,False,3,2017,8


### Contracts, competition actions and remarks
* Change columns to int where it makes sense

Remarks
* Again some missing values in contracts_signed and number_of_previous_teams, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before
* Outliers in draft_year, probably corresponding with old players in the first dataset
* Some crazy high outliers in salaries too
* After contracts are resolved, also worth to check contracts against the number of prevoius teams

## Performance

In [12]:
performance["goals"] = performance["goals"].astype("Int64")
performance["assists"] = performance["assists"].astype("Int64")
performance["num_of_shots"] = performance["num_of_shots"].astype("Int64")
performance["shot_attempts"] = performance["shot_attempts"].astype("Int64")
performance["high_danger_shots"] = performance["high_danger_shots"].astype("Int64")
performance["medium_danger_shots"] = performance["medium_danger_shots"].astype("Int64")
performance["low_danger_shots"] = performance["low_danger_shots"].astype("Int64")
performance["winning_goals"] = performance["winning_goals"].astype("Int64")
performance["power_play_goals"] = performance["power_play_goals"].astype("Int64")
performance["puck_touches"] = performance["puck_touches"].astype("Int64")
performance["puck_recoveries"] = performance["puck_recoveries"].astype("Int64")
performance["penalties_taken"] = performance["penalties_taken"].astype("Int64")
performance["goals_against_total"] = performance["goals_against_total"].astype("Int64")
performance["passes_attempted"] = performance["passes_attempted"].astype("Int64")
performance["passes_completed"] = performance["passes_completed"].astype("Int64")
performance["games_missed_due_to_injury"] = performance["games_missed_due_to_injury"].astype("Int64")

In [13]:
# Clean performance

performance

,international_id,goals,assists,num_of_shots,shot_speed,shot_attempts,shooting_percentage,high_danger_shots,medium_danger_shots,low_danger_shots,...,penalty_kill_time,penality_minutes,penalties_taken,time_between_penalties,goals_against_total,goals_against_average,passes_attempted,passes_completed,pass_completion_rate,games_missed_due_to_injury
0,6938,166,244,461,130.6172,480,12.6758,158,185,137,...,3.23,13.0,0,8.02,0,0.00,759,148,19.50,28
1,3924,98,166,459,76.4679,490,10.1533,142,186,162,...,3.47,2.0,1,1.14,0,0.00,456,354,77.63,21
2,3365,87,167,439,82.1817,449,33.3924,134,165,150,...,0.00,8.0,0,54.28,0,0.00,371,171,46.09,19
3,1284,93,155,447,73.0825,447,30.7875,141,136,170,...,1.81,16.0,1,8.89,0,0.00,411,137,33.33,17
4,5333,104,169,496,75.9673,503,42.2507,188,163,152,...,4.78,0.0,0,18.33,0,0.00,455,183,40.22,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1606,56,124,418,77.1258,460,27.2713,160,143,157,...,4.00,11.0,0,6.46,0,0.00,792,512,64.65,19
9996,9894,64,143,424,102.5506,433,24.2199,154,162,117,...,5.44,1.0,3,1.65,0,0.00,436,404,92.66,15
9997,5548,66,120,447,88.0325,470,47.7132,157,182,131,...,1.07,24.0,1,16.98,0,0.00,388,167,43.04,17
9998,2370,97,280,450,118.9816,476,52.6839,140,195,141,...,0.00,30.0,0,15.26,24,19.27,726,340,46.83,30


In [14]:
performance['passes_completed'] / performance['passes_attempted']

0       0.194993
1       0.776316
2       0.460916
3       0.333333
4       0.402198
          ...   
9995    0.646465
9996    0.926606
9997    0.430412
9998     0.46832
9999    0.495633
Length: 10000, dtype: Float64

### Performance actions and remarks

Remarks
* How is shooting_percentage calculated? The number doesn't make sense at all.
* danger shots add up to shot_attempts not num_of_shots.
* There are some people with save_percentages higher than 100.
* There are some people with higher power_play_time than time_on_ice. Could be time_on_ice is in hours, where power_play is in seconds but seems weird to me.
* Puck touches again don't make any sense, as 453 players have more goals than puck touches and 5932 have more shot attempts than puck touches.
* Puck recoveries also don't make any sense as there are 2650 players with more recoveries than puck touches...
* 3922 players have higher puck_possesion time than time_on_ice :)
* 841 player have a higher penalty kill time than time_on_ice...
* 5762 players with 0 penalties taken and non-0 penalty time.
* 5941 players with 0 penalties and non-0 time_between_penalties.
* 1314 players with higher goals agains on average than total goals against.
* 6513 players with higher passes attempted than puck touches,

TODO
* normalizing save_percentages (max 100) 
* assume that time_on_ice is in hours, and power_play_time is minutes and normalize into seconds 
* drop puck touches


In [15]:
# Clean medical information

# normalize height and weight?

# convert age to int

# remove percent symbol in body_fat_percentage 

# which columns are unnecessary?

medical_information

,medical_id,height,weight,age_in_years,body_fat_percentage,fitness_level,sprint_time,medical_information
0,368bcbfc-cc90-43da-841f-aa57d4d789c7,1.79m,82.4772kg,24.0,20.87%,Average,3.9727,NaN
1,f7164dc2-6646-420e-b6b5-42fc07c11f6d,180.1135cm,93.8985,29.0,23.38,NaN,4.1549,NaN
2,09c9de13-5d3a-4b75-a7ec-ce10bb32efdd,163.1483cm,79.2438,25.0,23.41,good,4.0825,NaN
3,be8969bf-6de2-49fd-b0e9-c014bdb970d6,1.82m,80.8913kg,29.0,21.28%,Good,3.5001,NaN
4,476cd0f0-0154-4e70-be7c-661007e7fcc4,167.7038cm,109.2016,25.0,31.71,excellent,3.7619,NaN
...,...,...,...,...,...,...,...,...
9995,d8686224-e95b-4da5-86a9-0f43a4f63f26,174.6442cm,90.3047,21.0,24.89,excellent,4.0910,NaN
9996,6d9916ac-307f-4ceb-8f7d-a284b76831bc,160.4894cm,84.1026,20.0,28.3,elite,4.3003,NaN
9997,bf144ec6-af6e-4a4f-9026-7f02999205d7,158.7063cm,77.7094,28.0,25.62,average,3.8575,NaN
9998,9ba4ccea-771d-4cfc-9172-d2bf922178f3,173.6693cm,80.162,31.0,25.19,Elite,3.6665,NaN
